In [25]:
import requests
import pandas as pd
import numpy as np

In [26]:
api_key='YOUR_API_KEY'

In [27]:

url = "https://newsapi.org/v2/everything"

war_keywords = [
    "war", "conflict", "military", "airstrike", "missile",
    "bomb", "attack", "invasion", "army", "troops",
    "defense", "clash", "border", "violence", "strike"
]
queries = [
    "war",
    "military conflict",
    "airstrike attack",
    "border clash",
    "missile strike",
    "army operation",
    "defense military",
    "troops conflict"
]

all_articles = []
news_list = []
for query in queries:
    for page in range(1, 6):

        params = {
            "q": query,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 20,
            "page": page,
            "apiKey": api_key
        }

        response = requests.get(url, params=params)
        data = response.json()

        articles = data.get("articles", [])
        all_articles.extend(articles)

for article in all_articles:
    news = {
        "title": article.get("title", ""),
        "content": article.get("description", ""),
        "source": article.get("source", {}).get("name", ""),
        "url": article.get("url", ""),
        "date": article.get("publishedAt", "")
    }
    news_list.append(news)

In [28]:

df = pd.DataFrame(news_list)

In [29]:
df.head(10)

""


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [15]:
df['content'][4]

KeyError: 'content'

In [ ]:
len((df))

In [ ]:
df = df[~df["title"].str.lower().str.contains("movie|film|trailer|review|box office")]

In [ ]:
len(df)

In [ ]:
from transformers import pipeline

# Load once (don’t put inside loop)
classifier = pipeline("zero-shot-classification")

def is_war_news(text):
    labels = ["war or military news", "not related"]

    result = classifier(text, labels)

    return result["labels"][0] == "war or military news"

In [ ]:
df = df[df["title"].apply(is_war_news)]

In [ ]:
print(len(df))


In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

df["title"] = df["title"].fillna('').str.lower()
df["content"] = df["content"].fillna('').str.lower()
df["source"] = df["source"].astype(str).str.lower()

df['combined_text'] = df['title'] + " " + df['content']

embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

In [ ]:
embeddings.shape


In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [ ]:
query = "pakistan and india at war"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding)

faiss.normalize_L2(query_embedding)#testing

In [ ]:
D, I = index.search(query_embedding, k=5)

In [ ]:
for i in I[0]:
    print(df.iloc[i]["title"])
    print(df.iloc[i]["source"])
    print(df.iloc[i]['url'])
    print("-----")

In [ ]:
!pip install transformers

In [ ]:
!pip install feedparser

In [ ]:
import numpy as np
import pandas as pd
import faiss
import feedparser
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder


model = SentenceTransformer('all-MiniLM-L6-v2')
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
df['content'] = df['content'].fillna('').astype(str).str.lower()
df['title'] = df['title'].fillna('').astype(str).str.lower()
df['date'] = df['date'].astype(str)

df['full_text'] = df['title'] + " " + df['content']

embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


war_keywords = [
    "war", "conflict", "attack", "military", "missile",
    "battle", "army", "troops", "airstrike", "bomb",
    "border", "clash", "violence", "defense", "ceasefire"
]
def expand_query(query):
    return query + " war conflict military attack"


def smart_filter(results, query):
    words = [w for w in query.split() if len(w) > 2]

    filtered = []

    for item in results:
        text = (item["title"] + " " + item["content"]).lower()


        if not any(k in text for k in war_keywords):
            continue

        if len(words) == 1:

            if words[0] in text:
                filtered.append(item)
        else:

            if all(word in text for word in words):
                filtered.append(item)

    return filtered


def clean_google_link(link):
    try:
        response = requests.get(link, allow_redirects=True, timeout=5)
        return response.url
    except:
        return link

def fetch_rss_news(query):

    query_formatted = query.replace(" ", "+")

    rss_urls = [
        f"https://news.google.com/rss/search?q={query_formatted}+war",
        f"https://news.google.com/rss/search?q={query_formatted}+conflict",
        f"https://news.google.com/rss/search?q={query_formatted}+military",

        "https://www.defensenews.com/arc/outboundfeeds/rss/?outputType=xml",
        "https://www.militarytimes.com/arc/outboundfeeds/rss/",
        "https://www.armytimes.com/arc/outboundfeeds/rss/"
    ]

    rss_data = []

    for url in rss_urls:
        feed = feedparser.parse(url)

        for entry in feed.entries:
            text = entry.title.lower()

            if any(k in text for k in war_keywords):
                rss_data.append({
                    "title": entry.title.lower(),
                    "content": entry.title,
                    "url": clean_google_link(entry.link),
                    "date": entry.published if "published" in entry else "",
                    "full_text": entry.title.lower()
                })

    seen = set()
    unique = []

    for item in rss_data:
        if item["url"] not in seen:
            unique.append(item)
            seen.add(item["url"])

    return unique[:40]

def get_top_matches(query):

    original_query = query.lower().strip()
    expanded_query = expand_query(original_query)

    print("Searching...")


    query_embedding = model.encode([expanded_query])
    query_embedding = np.array(query_embedding)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k=20)

    api_results = []

    for i in I[0]:
        if i < len(df):
            api_results.append({
                "title": str(df.iloc[i]['title']),
                "content": str(df.iloc[i]['content'])[:1000],
                "url": df.iloc[i]['url'],
                "date": df.iloc[i]['date']
            })


    rss_results = fetch_rss_news(original_query)

    api_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in api_results]
    api_scores = cross_model.predict(api_pairs) if api_pairs else []
    api_ranked = sorted(zip(api_scores, api_results), reverse=True)

    rss_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in rss_results]
    rss_scores = cross_model.predict(rss_pairs) if rss_pairs else []
    rss_ranked = sorted(zip(rss_scores, rss_results), reverse=True)

    combined = [item for _, item in (api_ranked[:5] + rss_ranked[:5])]

    filtered = smart_filter(combined, original_query)

    if not filtered:
        print("Not Found")
        return []

    final_results = filtered[:10]

    clean_results = []

    for item in final_results:
        try:
            parsed = pd.to_datetime(item["date"], errors='coerce')
            date_str = parsed.strftime("%Y-%m-%d") if not pd.isna(parsed) else str(item["date"])
        except:
            date_str = str(item["date"])

        clean_results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"],
            "date": date_str
        })

    if len(clean_results) >= 2:
        prompt = create_prompt(original_query, clean_results)
        summary = get_llm_response(prompt)

        print("\nSummary:\n")
        print(summary)

    print("\nSources:\n")

    for i, news in enumerate(clean_results):
        print(f"{i+1}. {news['title']}")
        print(news["url"])
        print("Date:", news["date"])
        print()


In [ ]:
!pip install Groq

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"]=getpass("Enter your Groq API key: ")

In [ ]:
from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def create_prompt(query, clean_results):

    prompt = f"""
You are a war news analyst.

User Query: {query}

Below are some news articles:

"""

    for i, news in enumerate(clean_results, 1):
        prompt += f"""
Article {i}:
Title: {news['title']}
Content: {news['content']}
Date: {news['date']}
"""

    prompt += """
Task:
- Summarize what is happening
- Explain in simple simple words
-Key insights:
- Keep answer short
"""

    return prompt

In [ ]:
def get_llm_response(prompt):

    print("LLM running...")

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    print("LLM done!")

    return response.choices[0].message.content

In [24]:
print(response.status_code)

401


In [ ]:
get_top_matches("")


In [ ]:
"""
Evaluation pipeline.


"""

import pandas as pd



def get_top_matches_eval(query, k=10):
    import numpy as np
    import faiss

    original_query = query.lower().strip()
    expanded_query = expand_query(original_query)

    query_embedding = model.encode([expanded_query])
    query_embedding = np.array(query_embedding)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k=20)

    api_results = []
    for i in I[0]:
        if i < len(df):
            api_results.append({
                "title": str(df.iloc[i]['title']),
                "content": str(df.iloc[i]['content'])[:1000],
                "url": df.iloc[i]['url'],
                "date": df.iloc[i]['date']
            })

    rss_results = fetch_rss_news(original_query)

    api_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in api_results]
    api_scores = cross_model.predict(api_pairs) if api_pairs else []
    api_ranked = sorted(zip(api_scores, api_results), reverse=True)

    rss_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in rss_results]
    rss_scores = cross_model.predict(rss_pairs) if rss_pairs else []
    rss_ranked = sorted(zip(rss_scores, rss_results), reverse=True)

    combined = [item for _, item in (api_ranked[:5] + rss_ranked[:5])]


    seen = set()
    unique_results = []

    for item in combined:
        if item["url"] not in seen:
            seen.add(item["url"])
            unique_results.append(item)

    filtered = smart_filter(unique_results, original_query)

    return [item["url"] for item in filtered[:k]]



def precision_at_k(retrieved, relevant, k):
    retrieved = list(dict.fromkeys(retrieved[:k]))
    relevant = set(relevant)

    if len(retrieved) == 0:
        return 0.0

    hits = len(set(retrieved) & relevant)

    return hits / len(retrieved)

def recall_at_k(retrieved, relevant, k):
    retrieved = list(dict.fromkeys(retrieved[:k]))
    relevant = set(relevant)

    if len(relevant) == 0:
        return 0.0

    hits = len(set(retrieved) & relevant)

    return hits / len(relevant)

def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)



TEST_QUERIES = {
    "iran and us": {
        "relevant_urls": ["https://www.cnbc.com/2026/06/27/tanker-struck-in-strait-of-hormuz-as-us-iran-tensions-escalate.html",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.aljazeera.com/news/2026/7/2/how-us-iran-war-may-push-gulf-countries-to-diversify-security-alliances",
"https://economictimes.indiatimes.com/news/defence/trump-threatens-to-annihilate-iran-after-new-exchange-of-attacks/articleshow/132044301.cms",
"https://news.google.com/rss/articles/CBMihAFBVV95cUxOVklRdE1jXzRId2ctSWRLWWFhRGtnOTNKZGN0dzh3QTVINTZhc1VNYUFudGpVOFJOTVhFbEQwbnBjQ2tXTUp6UEFBWUpXMkF1Y2tfT2dUSEItem9vTEQ3dS0ybWY1WF9RV1cyZXpLajF6ZThrVWRHak9JZTE4NE5lWFRocmE?oc=5&hl=en-US&gl=US&ceid=US:en"
        ]
    },
    "russia ukraine": {
        "relevant_urls": ["https://www.naturalnews.com/2026-07-03-civilian-toll-mounts-russia-ukraine-war-new-phase.html",
"https://cryptobriefing.com/russia-escalates-ukraine-conflict-with-2200-drones-1730-bombs-in-a-week/",
"https://cryptobriefing.com/russian-military-losses-reach-14m-facing-severe-attrition-in-ukraine-conflict/",
"https://news.google.com/rss/articles/CBMi6wFBVV95cUxOUlBBWmxTT0lPZHBvU1gzdmdIbF9seDVfTWlUWFlZZWhKeGJIWS1oNWpiSy04azR3R2haLVgxLXY1SV9iUGVmVW8xcEt0MFJrUERsLWtnMlowVmFaVTc3MTZZZURzV1N1S2prU09DM2RUckpKUFJhRmVncFltaUplRy1jWXBTNUdpcUhmRlY1RFJFc2pnNmdEY01MUDZOTXBiRUljaWtKbmJzbU9qT29NLTYzdFVMeVJ0UXQtNGJJRS1kSkhSX2Ftb2lWMGJCU3ZVbVJuakVtQzNYTnM4M2RZRFFrYUdYaU83NmhZ0gHwAUFVX3lxTE1xejlDMEtrMUFxU2JNV1JHUk9rMHlXMktmZzVvckpoV2NaU0FmU01ieDhuTEJqMWQzWmhTY1BiUWNldTNEak9SbmdyVnVHN19WaE01SlUwUzhfSjdBeHRBWWZOalN1aHhTSzMzcTllZ0RYRkZpYUxUS04wNHYxQ21idzMtNEhCV2NjRkxiVEg5cUwwZ0h1UTgyQlA3cnZUUTl4eUxKTFQ4aVdyWEFraFFvTklXWEZ3NDVlTXBlMkVqQm9iY1NUUzJ6QVd1WFJSeXRfQWlLQVRJZnhMWGpiT2o5V2VuS0dqdU5LamJzcmJTbg?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMiZEFVX3lxTFAyRk11NkNkWGtSZ2VCaVUyMTBab3BtOU8wVE9jYXZqa3dhTjFmVkNnalRISzhjVGltdXFiRFpEY2l0Z2lEdHlTNzAwc2tpREdScW4xM244QmNVTG53dnA3bFMwbzfSAXBBVV95cUxQMUJhZFdZU09tMVY1TjZXcm1VeElpTUxRaUlmSjFOUm14UEN0TlZGcFBKWmNqZXhPSlUxVzdYVEk2cTJzYUlrTlFWWmtnT3U5R1Ftb1RXU2hZV3NqZkpVUG9HaEZIUUNtVkpzdFRMSE5m?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMixAFBVV95cUxORzJ6cVRHTlpSM0dyc2toeTFncl9NN3RvSHNpYmc3bEhGX0dsbWljV0VQS1h4ZmFWODh2ZzN6QWdXUGtSdWtaQkwwZWk2cWpWMlY5Rk9HRjk0Sl9WYV9ISER0VEU5OUhfTnMwSDRHM1ZyaUR6M24yVG4xOGpoV3J5ZUxILVZyY0k3OUZiZDY4U0RrV3lhbTA3Z09wQnd1Rkxjcm9nc0V1ajRJRUxzQzBVRkRGN19YMjNLeFhZaXRvYUJzMl8t?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMid0FVX3lxTE1qa3hMcDFlQ3pPa0YyNllFUjFqZFdyNUNra3pnLVhGc3VsWWh0dDhuLUEyS3lCbXhfcGItZ0I3TGl3ZEo2YVR0cno1dXVoM2tWSzVPaXhHUTc5NHNPYWV1MGZuTXlVWVNOMXlrSjJGSnZvTDhOUC1B?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMipgFBVV95cUxOMnF6bnFNMU1jb3RIZWNEbVNRSU9IWTU3ektqWG53RHVibXZ5Y2ZQcDNRRUtXcWVHQ2g1RWFCSng0N2lXSzNlUm9iUnRnNmtHRGptMzkybVVtWnNpRXYyaHNNYmtHVXdyLVhneGlqWVoxV0RiMGRseWVvOFYxU1M4N2xyaXdmd3lRUlhEQmtLa29JZzZHa25wZGpzd19iQm9tLXB4N29B?oc=5&hl=en-US&gl=US&ceid=US:en"
]
    },
    "india pakistan tension": {
        "relevant_urls": ["https://news.google.com/rss/articles/CBMiWkFVX3lxTE9fcWRxY0VEMmdDbEkyaUFhNEw4QTY2elZuV192Rm1HRDBRZmEwcFhiS1VqajFPZzR6TUxNdzdTZGZ2QkNudmJDSk9IUERTS1RlOUlmUDQxTUswZw?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMi1wFBVV95cUxPUHM4TVJIbEo5NklVcEcwWl9ZRXRTWjFZLU50eXpHMnE4eG1mc1BRNDlrWVpXMWhjUHFvVlBucDVOLXhyRDFfYVhOOEpEZ1RwUjVVRHhtVnBVVTluR25zUmd5blJOc1BKRkJxZ2x5MmNFQU95d2dCa1h1enZSSXN2TmFBYlpYSE9HajN1bGlVU19CUXpweFhKclJ0UWtmblNNeGttWVBrWmhIdXNYSjlDM3dLejdudUJOUHI2LWlfZ3VjRzIxd3d4OTdBYTdqazBldEFoU0tnWdIB3AFBVV95cUxOXzBzOFBQX2xzUm4zLThXQ0FQc0YzYmM1aEtsMnFEeW4yLS1DcEpMdE10TjdWckF4aTI3c2dnLUpqTXpNZng0UEpUMlJhNHhwYUJXb2k1M2FNTjRNWmFqYjFuX0dEb21tS2pzWXFQU0FrSk0zX0NDWU9BTERxQ3pJbWxiTmtTTWpDeHdtQ0hPQmJIMWpNdWlUZXV5ZTNsbklyY2k0VGhRMjlWME5pcXo5UnVQcGNiQWVGVWx1d2o3VUFIazFJNHdxVjV5b0pucHhBeGVqZnFIVGh2YnBn?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMia0FVX3lxTE5qQ09NQmlVcjRBblEyTGVRNFVOaHVfa2x0a1hFU3NWUXJJcXBQOWhsQ1NGNF9JSEstZ005VFRNdlpERE5zbllXRDBIQ1J4RTJlaXlvQzdpZlQyWk40Ymk2RG9OS3FCZWxpUXVB?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMijwFBVV95cUxQaDRCalN1ejNfbmU4N21pUkp1WWlFWWMwdzRNdldKNmVlcXphVExpa0JLYTZ5TEFkOGdDOGdwX04yREtqMnVZeWEweTcxdGxQZTBXY1I2RHo3VmpGbVNaUjBfZk52cVlZYTB1ZnB1X3lfQ3I4REx4YnVmaUk2OTh6WmFaelpUZU1DT3dWbjZmNA?oc=5&hl=en-US&gl=US&ceid=US:en"
]
    },
    "China Taiwan": {
        "relevant_urls": ["https://cryptobriefing.com/china-raises-pressure-on-taiwan-with-expanded-coast-guard-patrols/",
"https://biztoc.com/x/18a3c581d605a8da",
"https://news.google.com/rss/articles/CBMijAFBVV95cUxPUF9iczBDVXlMQWRrTW9sbXZ4eUl6MUR3Zmo4RFBaenVMNjdPYkltSTVxSnhMSVlDdWdHN3ZGWDlxV3h0SENhUy04STJ2TlBkLTdZUEJqWW1tVDcwMFNfZlJzM0p1R3IwWUVNZ1FiVXd4cWxvV3VQQzZaeE9va0NxYXFRV1pCMEhxb1g5Sg?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMirgFBVV95cUxOSGtraEdWMTZNSHNKVUpvc0NndFI5S3ZVVEJieFV0ZHhJY1MzTTV0bWVrZmVscnRUUkdRUmlieU1zU0d2WkRybEVMTzYydjA4U05fSTBiR0VCekxKVGF2bTZYNXlDWE5iUUFMSUpMUzN0UjNvTlpwRFRQSDZiOVRRS2lULW55N2V0U0prZ2NITUZEMXZGcEpIMS16akozdU5yZGdjam9rWmVvWHZUVUE?oc=5&hl=en-US&gl=US&ceid=US:en:",
"https://news.google.com/rss/articles/CBMijgFBVV95cUxNeUJJdTZGN3NGYlVFelV0eThSTUx2RzMzTWlyT1ZXbi1hOXVPQWZsYzNuTXVlcVhzN0VSRlpMYUJRWF9IU3VsQzRKdjdyZHpoTVZpYVQ4UElwRkVSVjYwaDlxY0NNS0R1ZW9fZ0R3WHRXZ19DTktYQm9kd1BCTlltdXdKaUZEME0zYVVOdjF3?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMijAFBVV95cUxPLTlrVzlFeHcxT1dmU0JCYnFiUEpjTnZYNDhLMm9xV0ZIZ0VqQUdNNENPQVBXdWpNdTFxMTNYVFNMa29iTkI2X1R1V2EyOUhLeFJEZlpBd1BKRzdfZS1DbWJUUFpyYXdiRlg2RVdINVNsN1R1T0FPaFhnSGhnYWk4dGtQNmcwUkk3VW10UA?oc=5&hl=en-US&gl=US&ceid=US:en",
"https://news.google.com/rss/articles/CBMijAFBVV95cUxNRGlMNUc1OHdwcTlCdWVzZXZfNHVQLUg5dTIzTkMyY0lWOUc5MFpZeGhuTnc2MWRkSkxvaEJPSnY4a29YTHZCbXpGWFdpZXNKNEhwc2JWY255SFNGT2VEN3hUeGZzYjVNdkdPRWFIOWpKZjhEUDFVS1liRVFqd0oxTC1wZ3k5ZXFreFplOA?oc=5&hl=en-US&gl=US&ceid=US:en "]
    },"NATO Russia": {
        "relevant_urls": ["https://www.cnbc.com/2026/06/27/tanker-struck-in-strait-of-hormuz-as-us-iran-tensions-escalate.html",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.aljazeera.com/news/2026/7/2/how-us-iran-war-may-push-gulf-countries-to-diversify-security-alliances",
"https://economictimes.indiatimes.com/news/defence/trump-threatens-to-annihilate-iran-after-new-exchange-of-attacks/articleshow/132044301.cms",
"https://news.google.com/rss/articles/CBMihAFBVV95cUxOVklRdE1jXzRId2ctSWRLWWFhRGtnOTNKZGN0dzh3QTVINTZhc1VNYUFudGpVOFJOTVhFbEQwbnBjQ2tXTUp6UEFBWUpXMkF1Y2tfT2dUSEItem9vTEQ3dS0ybWY1WF9RV1cyZXpLajF6ZThrVWRHak9JZTE4NE5lWFRocmE?oc=5&hl=en-US&gl=US&ceid=US:en"
        ]
    },"Ukraine drones": {
        "relevant_urls": ["https://www.cnbc.com/2026/06/27/tanker-struck-in-strait-of-hormuz-as-us-iran-tensions-escalate.html",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.aljazeera.com/news/2026/7/2/how-us-iran-war-may-push-gulf-countries-to-diversify-security-alliances",
"https://economictimes.indiatimes.com/news/defence/trump-threatens-to-annihilate-iran-after-new-exchange-of-attacks/articleshow/132044301.cms",
"https://news.google.com/rss/articles/CBMihAFBVV95cUxOVklRdE1jXzRId2ctSWRLWWFhRGtnOTNKZGN0dzh3QTVINTZhc1VNYUFudGpVOFJOTVhFbEQwbnBjQ2tXTUp6UEFBWUpXMkF1Y2tfT2dUSEItem9vTEQ3dS0ybWY1WF9RV1cyZXpLajF6ZThrVWRHak9JZTE4NE5lWFRocmE?oc=5&hl=en-US&gl=US&ceid=US:en"
        ]
    },"Myanmar conflict": {
        "relevant_urls": ["https://www.cnbc.com/2026/06/27/tanker-struck-in-strait-of-hormuz-as-us-iran-tensions-escalate.html",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.aljazeera.com/news/2026/7/2/how-us-iran-war-may-push-gulf-countries-to-diversify-security-alliances",
"https://economictimes.indiatimes.com/news/defence/trump-threatens-to-annihilate-iran-after-new-exchange-of-attacks/articleshow/132044301.cms",
"https://news.google.com/rss/articles/CBMihAFBVV95cUxOVklRdE1jXzRId2ctSWRLWWFhRGtnOTNKZGN0dzh3QTVINTZhc1VNYUFudGpVOFJOTVhFbEQwbnBjQ2tXTUp6UEFBWUpXMkF1Y2tfT2dUSEItem9vTEQ3dS0ybWY1WF9RV1cyZXpLajF6ZThrVWRHak9JZTE4NE5lWFRocmE?oc=5&hl=en-US&gl=US&ceid=US:en"
        ]
    },"Yemen Houthis": {
        "relevant_urls": ["https://www.cnbc.com/2026/06/27/tanker-struck-in-strait-of-hormuz-as-us-iran-tensions-escalate.html",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.foxnews.com/politics/how-iran-attacks-forcing-pentagon-rethink-its-decades-old-middle-east-base-strategy",
"https://www.aljazeera.com/news/2026/7/2/how-us-iran-war-may-push-gulf-countries-to-diversify-security-alliances",
"https://economictimes.indiatimes.com/news/defence/trump-threatens-to-annihilate-iran-after-new-exchange-of-attacks/articleshow/132044301.cms",
"https://news.google.com/rss/articles/CBMihAFBVV95cUxOVklRdE1jXzRId2ctSWRLWWFhRGtnOTNKZGN0dzh3QTVINTZhc1VNYUFudGpVOFJOTVhFbEQwbnBjQ2tXTUp6UEFBWUpXMkF1Y2tfT2dUSEItem9vTEQ3dS0ybWY1WF9RV1cyZXpLajF6ZThrVWRHak9JZTE4NE5lWFRocmE?oc=5&hl=en-US&gl=US&ceid=US:en"
        ]
    },

}

K = 10  # evaluate top-10 results, since these are broad topical queries



def run_evaluation():
    rows = []

    for query, gt in TEST_QUERIES.items():
        relevant = set(gt["relevant_urls"])
        if not relevant:
            print(f"[SKIPPED] '{query}' has no ground truth URLs yet -- fill these in first.")
            continue

        retrieved = get_top_matches_eval(query, k=K)

        p = precision_at_k(retrieved, relevant, K)
        r = recall_at_k(retrieved, relevant, K)
        f1 = f1_score(p, r)


        false_positives = [u for u in retrieved if u not in relevant]

        rows.append({
            "query": query,
            f"precision@{K}": round(p, 2),
            f"recall@{K}": round(r, 2),
            "f1": round(f1, 2),
            "num_retrieved": len(retrieved),
            "num_false_positives": len(false_positives),
        })

    if not rows:
        print("\nNo queries evaluated -- add ground truth URLs to TEST_QUERIES first.")
        return None

    results_df = pd.DataFrame(rows)
    print("\n" + results_df.to_string(index=False))

    print(f"\nAverage Precision@{K}: {results_df[f'precision@{K}'].mean():.2f}")
    print(f"Average Recall@{K}: {results_df[f'recall@{K}'].mean():.2f}")
    print(f"Average F1: {results_df['f1'].mean():.2f}")

    return results_df


if __name__ == "__main__":
    run_evaluation()

In [ ]:
results = get_top_matches_eval("NATO Russia", k=10)

for url in results:
    print(url)